<a href="https://colab.research.google.com/github/Shanmukh-dev/Custom-GPT/blob/main/Tara_N1_Instruction_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import tiktoken
import torch
from torch import nn
import datasets


tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


## Downloading required Weights

In [ ]:
engine_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")

with open("model.py", "w") as f:
    f.write(engine_res.text)
    # print(engine_res.text)
print("Downloaded model.py")

# Downloading train_utils.py
train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")

with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
    # print(engine_res.text)
print("Downloaded train_utils.py")

data_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")

with open("data.py", "w") as f:
    f.write(data_res.text)
    # print(engine_res.text)
print("Downloaded data.py")

#Downloading custom_gpt_v2.pth weights
wt_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain_v2.pth")
with open("tara_n1_pretrain_v2.pth", "wb") as f:
    f.write(wt_res.content)
print("Downloaded modle weights")


In [3]:
from model import *
from train_utils import *
from data import *


config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=4,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

## Downloading the data

In [ ]:
dataset_res = requests.get("https://raw.githubusercontent.com/Shanmukh-dev/Custom-GPT/refs/heads/main/Data/tara_n1_sft_100k.json")
with open("sft_dataset", "w") as f:
  f.write(dataset_res.text)

## Dataset prep

In [4]:
import json

with open("Data/tara_n1_sft_100k.json", encoding="utf-8") as f:
  examples = json.load(f)

examples = list(examples)

In [5]:
# from _data import create_shards_dataloaders
train_dataloader, test_dataloader = create_sft_dataloaders(examples, tokenizer, 0.9, config.block_size, config.batch_size)

Total examples:  100000
Train examples:  13350905
Test examples:  1474473


In [15]:
# from torch.utils import grad

model = CustomGPT(config)

state_dict = torch.load("Models/tara_n1_pretrain_v2.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}

model.load_state_dict(state_dict)


model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

calc_params(model)

Total Parameters: 30,783,057
Trainable Parameters: 30,783,057


In [16]:
from tqdm.auto import tqdm

steps = 3260


train_iter = iter(train_dataloader)
avg_train_loss = 0
test_count = 500

for step in tqdm(range(1, steps+1)):
    model.train()
    if not train_iter:
        train_iter = train_iter

    train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)

    avg_train_loss += train_loss

    if step%test_count == 0:
        avg_train_loss /= test_count
        test_loss = test_step(model, test_dataloader, 20, loss_fn, device)

        print(f"Step: {step} | Train Loss: {avg_train_loss} | Test Loss: {test_loss}")
        avg_train_loss = 0

  0%|          | 0/3260 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Step: 500 | Train Loss: 2.6699578862190245 | Test Loss: 2.388411247730255


  0%|          | 0/20 [00:00<?, ?it/s]

Step: 1000 | Train Loss: 2.4709359686374666 | Test Loss: 2.4516335248947145


...

Step: 2000 | Train Loss: 2.3647680819034576 | Test Loss: 2.4582007765769958


  0%|          | 0/20 [00:00<?, ?it/s]

Step: 2500 | Train Loss: 2.3474279062747954 | Test Loss: 2.4198478400707244


  0%|          | 0/20 [00:00<?, ?it/s]

Step: 3000 | Train Loss: 2.3292625207901 | Test Loss: 2.3397513389587403


In [24]:
model.eval()

query = "What is your name?"
context = torch.tensor(tokenizer.encode(f"<user>{query}</user>\n"), dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = model.generate(context)

print(tokenizer.decode(generated[0].tolist()).split("<|endoftext|>")[0].replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">"))


<user>What is your name?</user>
<think>The user references their name to the company name from the perspective Purple < "How can I assist you if it works?"</user>
<think>I should answer the user's request clearly and directly.</think>
<assistant>The logo was extended and updated on building and what it was called during the application's Jokes script during the Jokes in apparent support for women and in multinationals and micrograms. By targeting the Joint Economic Capability Agreement, the main target material for replering workers is to use databases in most companies, Hospitals, and high risk societies to show which distribution they should also be able to achieve their goals.

Purple sales have obtained, leading to increasing emigration and leading to post-war transport. Letters of electric resource managers generally use apostools of techniques and polytechnic to attract engineers and make wells that can provide diversified job enthrust to the labor of cultivated gold mines. Prote

In [21]:
torch.save(model.state_dict(), "Models/tara_n1_it_v1.pth")